In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("EcritureCassandra") \
    .config("spark.jars.packages", "com.datastax.spark:spark-cassandra-connector_2.12:3.4.1") \
    .config("spark.cassandra.connection.host", "cassandra") \
    .config("spark.sql.extensions", "com.datastax.spark.connector.CassandraSparkExtensions") \
    .getOrCreate()

indicateur_compagnie = spark.read.parquet("/home/jovyan/data/indicateur_compagnie.parquet")
indicateur_aeroport = spark.read.parquet("/home/jovyan/data/indicateur_aeroport.parquet")
indicateur_jour = spark.read.parquet("/home/jovyan/data/indicateur_jour.parquet")
indicateur_aeroport_jour = spark.read.parquet("/home/jovyan/data/indicateur_aeroport_jour.parquet")

print("Indicateurs rechargés :")
indicateur_compagnie.show(3)
indicateur_aeroport.show(3)
indicateur_jour.show(3)
indicateur_aeroport_jour.show(3)

Indicateurs rechargés :
+-------------+-------+----------+------------+-----------+
|UniqueCarrier|nb_vols|nb_retards|retard_moyen|taux_retard|
+-------------+-------+----------+------------+-----------+
|           YV|   5111|      3772|        53.9|       73.8|
|           OH|   4096|      3019|       51.26|      73.71|
|           B6|   4268|      2940|       56.33|      68.88|
+-------------+-------+----------+------------+-----------+
only showing top 3 rows

+------+-------+----------+------------+-----------+
|Origin|nb_vols|nb_retards|retard_moyen|taux_retard|
+------+-------+----------+------------+-----------+
|   ILM|     60|        53|       68.03|      88.33|
|   EGE|     71|        60|      107.86|      84.51|
|   CHA|     82|        67|       78.43|      81.71|
+------+-------+----------+------------+-----------+
only showing top 3 rows

+---------+-------+----------+------------+-----------+--------+
|DayOfWeek|nb_vols|nb_retards|retard_moyen|taux_retard|nom_jour|
+----

In [7]:
# Table 1 - par compagnie
indicateur_compagnie.selectExpr(
    "UniqueCarrier as compagnie", "nb_vols", "nb_retards", "retard_moyen", "taux_retard"
).write \
    .format("org.apache.spark.sql.cassandra") \
    .mode("append") \
    .options(table="retards_par_compagnie", keyspace="vols") \
    .save()
print(" retards_par_compagnie écrite.")

 retards_par_compagnie écrite.


In [8]:
# Table 2 - par aéroport
indicateur_aeroport.selectExpr(
    "Origin as aeroport", "nb_vols", "nb_retards", "retard_moyen", "taux_retard"
).write \
    .format("org.apache.spark.sql.cassandra") \
    .mode("append") \
    .options(table="retards_par_aeroport", keyspace="vols") \
    .save()
print(" retards_par_aeroport écrite.")

 retards_par_aeroport écrite.


In [9]:
# Table 3 - par jour
indicateur_jour.selectExpr(
    "DayOfWeek as jour_semaine", "nom_jour", "nb_vols", "nb_retards", "retard_moyen", "taux_retard"
).write \
    .format("org.apache.spark.sql.cassandra") \
    .mode("append") \
    .options(table="retards_par_jour", keyspace="vols") \
    .save()
print(" retards_par_jour écrite.")

 retards_par_jour écrite.


In [10]:
# Table 4 - par aéroport et jour 
indicateur_aeroport_jour.selectExpr(
    "Origin as aeroport", "DayOfWeek as jour_semaine", "nom_jour",
    "nb_vols", "nb_retards", "retard_moyen", "taux_retard"
).write \
    .format("org.apache.spark.sql.cassandra") \
    .mode("append") \
    .options(table="retards_par_aeroport_jour", keyspace="vols") \
    .save()
print(" retards_par_aeroport_jour écrite.")

 retards_par_aeroport_jour écrite.
